# Day 42: Final ML Evaluation Review

YBI Foundation — Python AI & Data Science Internship

**Project:** AI-Based Student Performance Prediction System

## Objective

Audit model-selection evidence and reproduce historical metrics with an explicit baseline and uncertainty estimate.

## Theory

Macro F1 weights all classes equally, which matters for the imbalanced target. A majority baseline provides context. A stratified bootstrap measures conditional variability of historical evaluation records; it does not create independent validation.

## Prerequisites

Install `requirements-dev.txt` (and `requirements-api.txt` for API work) in the active Python environment. Use the repository checkout and the saved Day 17 model. The setup below locates or clones the repository for Colab. Run cells in order.


In [1]:
from pathlib import Path
import os, sys, subprocess, json
root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if not (root / "src").exists():
    target = root / "IYB_INTERNSHIP"
    if not target.exists():
        subprocess.run(["git", "clone", "https://github.com/lightcode01-oss/IYB_INTERNSHIP.git", str(target)], check=True)
    root = target
os.chdir(root)
sys.path.insert(0, str(root))
print("Project:", root.name)


Project: IYB_INTERNSHIP


Audit the training-CV selection and every saved test prediction.

In [2]:
from src.final_evaluation import review_evaluation
result=review_evaluation()
Path("reports/final_evaluation.json").write_text(json.dumps(result,indent=2))
print(json.dumps({k:v for k,v in result.items() if k not in ["classification_report","confusion_matrix"]},indent=2))

{
  "model_id": "1bedf35d0d56c4f0",
  "selected_model": "Decision Tree",
  "training_cv_macro_f1": 0.5236814699741081,
  "training_cv_std": 0.03894230481396425,
  "historical_test_rows": 130,
  "metrics": {
    "accuracy": 0.5384615384615384,
    "macro_precision": 0.4447496947496948,
    "macro_recall": 0.46288156288156285,
    "macro_f1": 0.44977366255144036
  },
  "majority_baseline_macro_f1": 0.2616822429906542,
  "conditional_bootstrap_macro_f1_95_percentile": [
    0.34722657869533885,
    0.5332758797065215
  ],
  "bootstrap_repeats": 500,
  "bootstrap_seed": 42,
  "class_order": [
    "Low",
    "Medium",
    "High"
  ],
  "limitations": [
    "Historical test set reused during earlier exploration; not fresh external validation.",
    "Bootstrap holds class counts fixed and conditions on this fitted model; it does not account for training or model-selection uncertainty.",
    "Small observational dataset from two Portuguese schools; no causal or educational decision validity.",

Write the human-readable evaluation and explanation reports from actual outputs.

In [3]:
import pandas as pd
report=f"# Final model evaluation\n\nSelected model: {result['selected_model']}. Training CV Macro F1: {result['training_cv_macro_f1']:.6f} (fold SD {result['training_cv_std']:.6f}). Selection was verified against the training-only tuning table.\n\nHistorical evaluation: {result['historical_test_rows']} records.\n\n"
report+=pd.DataFrame([result["metrics"]]).to_markdown(index=False)
report+=f"\n\nMajority baseline Macro F1: {result['majority_baseline_macro_f1']:.6f}. Conditional stratified bootstrap 95% percentile interval: {result['conditional_bootstrap_macro_f1_95_percentile']}.\n\n"
report+=pd.DataFrame(result["classification_report"]).T.to_markdown()
report+="\n\n## Limitations\n\n"+"\n".join("- "+x for x in result["limitations"])
Path("reports/FINAL_MODEL_EVALUATION.md").write_text(report)
importance=pd.read_csv("reports/day_24/permutation_importance.csv")
explanation="# Explainability report\n\nRaw-column permutation importance uses the saved pipeline, Macro F1, 15 shuffles and seed 42 on the historical 130-row evaluation set. Positive values mean shuffling reduced this model score. Negative values mean shuffling improved the sample score. Neither indicates a causal effect on students.\n\n"+importance.head(10).to_markdown(index=False)
explanation+="\n\nNative attribution tables, plots and individual tree paths are in reports/day_24/. Seed variability is in reports/explainability_validation.csv. Correlation, small sample size and test reuse limit interpretation. No educational recommendation is derived from these associations."
Path("reports/FINAL_EXPLAINABILITY.md").write_text(explanation)
print("Evaluation and explanation reports written from verified outputs")

Evaluation and explanation reports written from verified outputs


## Findings and interpretation

The saved tree, hyperparameters, training-CV ranking, split indices and all test predictions agree with the historical artifacts. The reports state measured performance, the majority baseline and bootstrap scope.

## Limitations

Repeated test inspection limits generalization claims. The interval conditions on fixed predictions and class counts and must not be presented as uncertainty for a future school population.

## Development log

Implemented and ran the cells above during the repair and extension session on 2 November 2026. The saved outputs are produced by the kernel.

## Conclusion

Audit model-selection evidence and reproduce historical metrics with an explicit baseline and uncertainty estimate.

## Next-day plan

Run final integration tests and rerun the extension notebooks.